In [ ]:
%reload_ext autoreload
%autoreload 2

from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

import os
import logging
import yaml
import ipynbname

import pandas as pd

import ferdinand.utils as utl   
import ferdinand.sqlite_utils as sqlutl
import ferdinand.preprocess_and_qc as ppqc

from pathlib import Path
from types import SimpleNamespace


# Purpose and Documentation

## Purpose

This notebook applies center-specific preprocessing to downloaded IMPC radiographs, preparing them for downstream analysis or feature extraction. Typical steps include contrast enhancement (CLAHE), noise reduction, removal of acquisition markers, cropping or padding to a consistent image shape, and recording preprocessing status in the SQLite metadata.

## Location

_&lt;project_dir&gt;notebooks/300_preprocess_dataset.ipynb_

## Prerequisites

- A valid runtime configuration file (_../config/config.yml_) with correctly defined _centers_, _run_name_, _parameters_, and _directories_
- Successful execution of the template setup cells from _NNN_notebook_template.ipynb_
- Existing _../data/_ and _../reports/_ directories 
- Existing SQLite database _../data/&lt;run_name&gt;/&lt;all_data_dir&gt;/IMPC_radiographs.db_
- Image files have already been downloaded and are present in _../data/&lt;run_name&gt;/&lt;images_dir&gt;_

# Definition / Settings

This section initializes the notebook environment: 
- loads the runtime configuration from _../config/config.yml_
- creates the required directory structure
- sets up logging and the SQLite connection

**Do not proceed past this section** </br>
Missing paths and an uninitialized database connection will cause subsequent cells to crash.

## General settings

In [ ]:
# load the configuration file
config_path = Path('../config/config.yml')
with open(config_path, 'r') as file:
    config = yaml.safe_load(file)

print(pd.DataFrame(config['settings'].items(), columns=['configuration parameter', 'set to']).to_markdown(tablefmt="pretty", index=False))

## Set up runtime paths

In [ ]:
# create directories as per configuration
dirs = SimpleNamespace(**config['data_directories'])
for key, value in vars(dirs).items():
    dirs.__dict__[key] = os.path.join(config['project_root'], config['data_dir'], config['run_name'], value)
    os.makedirs(dirs.__dict__[key], exist_ok=True)
    if os.path.exists(dirs.__dict__[key]):
        print(f" Directory created: {dirs.__dict__[key]}")
    else:
        print(f" ERROR: Directory does not exist: {dirs.__dict__[key]}")

## Store runtime configuration when necessary

In [ ]:
# copy the current '../config/config.yml' to '../reports/'
dest_filename = f"config_{config['run_name']}.yml"
dest_path = os.path.join(config['project_root'], config['reports_dir'], config['run_name'], dest_filename)
if not os.path.exists(dest_path): 
    commmand_string = f"cp {config_path} {dest_path}"
    os.system(commmand_string)

    if os.path.exists(dest_path): 
        print(f" Current runtime configuration \'{config_path}' saved to \'{dest_path}\'")
    else:
        print(f" ERROR: could not save configuration file to \'{dest_path}\'")

## Set up logger

In [ ]:
# setup logging
notebook_name = ipynbname.name() 
notebook_idx = notebook_name.split('_')[0]

log_path = os.path.join(config['project_root'], config['reports_dir'], config['run_name'], 'logs')
os.makedirs(log_path, exist_ok=True)

log_file = os.path.join(log_path, f"{notebook_name}.log")
print(f"Log file: {log_file}")

try:
    logger = utl.setup_logger(log_file, log_level=logging.INFO)
    print("Logger successfully set up.")
except Exception as e:
    print(f"Error setting up logger: {e}")

## Set up SQLite connection

In [ ]:
# expected SQLlite database file
sqlite_file = os.path.join(dirs.all_data_dir, f'{config["sqlite_db"]}.db')

# if already exists, connect to it
if os.path.exists(sqlite_file):
    conn = sqlutl.connect_db(sqlite_file)
    if conn is not None:
        print(f"connected to database {sqlite_file}\n")
else: 
    print(f" ERROR: No SQLite database found!")

# Image preprocessing

## Preprocessing a specific image

In [ ]:
selected_image = utl.select_jpegs_by_column(conn=conn, 
                                            db_table='impc_data', 
                                            column='omero_id', 
                                            value=85539, 
                                            source_path=dirs.images_dir)
print(selected_image)
utl.plot_image_grid(image_files=selected_image, cols=5)

In [ ]:
center = 'RBRC'
logger.info(f"preprocessing images from center {center} ...")
ppqc.preprocess_images(conn=conn, 
                       db_table='impc_data',
                       center=center,
                       image_files=selected_image,
                       source_path=dirs.images_dir,
                       target_path=dirs.preprocessed_dir,
                       logger=logger)

## Preprocessing a handful of images per center

In [ ]:
# specify list of centers
my_centers = config['centers']
print(f"Centers to be processed: {my_centers}")

In [ ]:
# randomly select images from each given center
rnd_selected_images = utl.plot_random_image_grid(conn=conn, 
                                                 db_table='impc_data', 
                                                 image_path=dirs.images_dir,
                                                 no_of_images=20,
                                                 centers=my_centers,
                                                 cols=5)

In [ ]:
# preprocess the randomly selected images
for center in my_centers:
    logger.info(f"preprocessing images from center {center} ...")
    ppqc.preprocess_images(conn=conn, 
                           db_table='impc_data',
                           center=center,
                           image_files=rnd_selected_images[center],
                           source_path=dirs.images_dir,
                           target_path=dirs.preprocessed_dir,
                           logger=logger)

## Center-by-center batch preprocessing of all available images

In [ ]:
# specify list of centers
my_centers = config['centers']
print(f"Centers to be processed: {my_centers}")

In [ ]:
# preprocess all images from each center
for center in my_centers:
    
    jpegs_by_center = utl.select_jpegs(conn=conn, 
                                      db_table='impc_data',
                                      image_path=dirs.images_dir,
                                      center=center)
    print(f" number of jpegs from {center}: {len(jpegs_by_center)}")

    ppqc.preprocess_images(conn=conn,
                           db_table='impc_data', 
                           center=center,
                           image_files=jpegs_by_center,
                           source_path=dirs.images_dir,
                           target_path=dirs.preprocessed_dir,
                           logger=logger)

# Create thumbnail gallery

In [ ]:
# select rows from the database where preprocessed is yes
rows = sqlutl.select_rows_by_column(conn, 'impc_data', 'preprocessed', 'yes')

In [ ]:
# specify list of centers
my_centers = config['centers']
print(f"Centers to be processed: {my_centers}")

## Create thumbnails for selected rows

In [ ]:
# create thumbnails for all preprocessed images
utl.create_and_save_thumbnails(
    data=rows,
    centers=my_centers,
    path2images=dirs.preprocessed_dir,
    path2thumbnails=dirs.preprocessed_thumbnails_dir
)

In [ ]:
utl.create_and_save_thumbnails(
    data=rows,
    centers=['JAX'],
    path2images=dirs.preprocessed_dir,
    path2thumbnails=dirs.preprocessed_thumbnails_dir,
    size=(128,128)
)

## Create HTML galleries with generated thumbnails

In [ ]:
# create an HTML gallery directory
gallery_dir = os.path.join(config['project_root'], config['reports_dir'], config['run_name'], 'thumbnails_gallery')
os.makedirs(gallery_dir, exist_ok=True)

In [ ]:
# create HTML gallery of preprocessed images
utl.create_thumbnail_gallery(
    data=rows,
    html_file_name=f'{notebook_idx}_all_preprocessed_images.html',
    title='Overview of Preprocessed Images',
    html_dir=gallery_dir,
    thumbnails_dir=dirs.preprocessed_thumbnails_dir,
    source_dir=dirs.preprocessed_dir)

In [ ]:
# create HTML galleries of preprocessed images for each center
for center in my_centers:
    center_df = rows[rows['center'] == center]
    if not center_df.empty:
        center_stripped = center.replace(" ", "_")
        center_gallery_dir = os.path.join(gallery_dir, center_stripped)
        os.makedirs(center_gallery_dir, exist_ok=True)
        utl.create_thumbnail_gallery(
            data=center_df,
            html_file_name=f'{notebook_idx}_all_preprocessed_images_{center_stripped}.html',
            title=f'Overview of Preprocessed Images from {center}',
            html_dir=center_gallery_dir,
            thumbnails_dir=dirs.preprocessed_thumbnails_dir,
            source_dir=dirs.preprocessed_dir)